In [2]:
import torch 
import json 
import trimesh 
import os 
from torch.utils.data import Dataset, DataLoader 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import DictConfig


## Dataset

In [ ]:
class ScanReferDataset(Dataset):
    def __init__(self, scanrefer_data_path, scannet_dir, num_points=40000):
        # Load the ScanRefer JSON
        with open(scanrefer_data_path, 'r') as f:
            self.scanrefer_data = json.load(f)
            
        self.scannet_dir = scannet_dir
        self.num_points = num_points

    def __len__(self):
        return len(self.scanrefer_data)

    def __getitem__(self, idx):
        item = self.scanrefer_data[idx]
        scene_id = item["scene_id"]
        target_obj_id = str(item["object_id"]) 
        
        # 1. Load the 3D Point Cloud (.ply)
        ply_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}_vh_clean_2.ply")
        mesh = trimesh.load(ply_path, process=False)
        
        points = np.array(mesh.vertices) 
        colors = np.array(mesh.visual.vertex_colors[:, :3]) / 255.0 
        
        # 2. Extract Ground Truth from JSONs using raw points
        agg_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}.aggregation.json")
        with open(agg_path, 'r') as f:
            agg_data = json.load(f)
            
        target_segments = []
        for seg_group in agg_data['segGroups']:
            if str(seg_group['objectId']) == target_obj_id:
                target_segments = seg_group['segments']
                break
                
        segs_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}_vh_clean_2.0.010000.segs.json")
        with open(segs_path, 'r') as f:
            segs_data = json.load(f)
            
        seg_indices = np.array(segs_data['segIndices'])
        valid_vertex_mask = np.isin(seg_indices, target_segments)
        target_points = points[valid_vertex_mask]

        # -------------------------------------------------------------
        # STEP A: Calculate the raw Bounding Box FIRST
        # -------------------------------------------------------------
        if len(target_points) > 0:
            gt_min = np.min(target_points, axis=0)
            gt_max = np.max(target_points, axis=0)
            gt_box_center = (gt_max + gt_min) / 2.0
            gt_box_size = gt_max - gt_min
        else:
            gt_box_center = np.zeros(3)
            gt_box_size = np.ones(3) * 1e-6 

        # -------------------------------------------------------------
        # STEP B: Apply Normalization to EVERYTHING
        # -------------------------------------------------------------
        # 1. Calculate normalization params from the full scene
        min_xyz = np.min(points, axis=0)
        max_xyz = np.max(points, axis=0)
        range_xyz = max_xyz - min_xyz
        range_xyz[range_xyz == 0] = 1.0

        # 2. Normalize the full scene points
        points = (points - min_xyz) / range_xyz

        # 3. Apply the EXACT same shift and scale to the GT Box Center
        gt_box_center = (gt_box_center - min_xyz) / range_xyz

        # 4. Apply ONLY the scale to the GT Box Size (size doesn't shift)
        gt_box_size = gt_box_size / range_xyz
        # -------------------------------------------------------------
            
        # 3. Downsample the Point Cloud
        point_cloud = np.concatenate([points, colors], axis=1)
        
        if point_cloud.shape[0] > self.num_points:
            choices = np.random.choice(point_cloud.shape[0], self.num_points, replace=False)
            point_cloud = point_cloud[choices, :]
        else:
            padding = np.zeros((self.num_points - point_cloud.shape[0], 6))
            point_cloud = np.vstack((point_cloud, padding))
            
        text_tokens = item["token"]
        raw_text = " ".join(text_tokens)
        
        return {
            "point_cloud": torch.tensor(point_cloud, dtype=torch.float32),
            "gt_box_center": torch.tensor(gt_box_center, dtype=torch.float32),
            "gt_box_size": torch.tensor(gt_box_size, dtype=torch.float32),
            "text": raw_text
        }
        
    def visualize_point_cloud(self, idx):
        
        datapoint = self.__getitem__(idx)
        
        # Convert tensors back to numpy arrays for plotting
        point_cloud = datapoint["point_cloud"].numpy()
        gt_box_center = datapoint["gt_box_center"].numpy()
        gt_box_size = datapoint["gt_box_size"].numpy()
        text = datapoint["text"]
        
        # Extract coordinates and colors
        points = point_cloud[:, :3]
        colors = point_cloud[:, 3:6]
        
        # Filter out the padding (rows that are entirely zero)
        mask = np.any(points != 0, axis=1)
        points = points[mask]
        colors = colors[mask]
        
        # Create a 3D plot
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Plot the point cloud
        # Matplotlib can be sluggish with 40k points. Slicing [::2] or [::4] 
        # subsamples the points for smoother rendering if needed.
        ax.scatter(points[::2, 0], points[::2, 1], points[::2, 2], 
                   c=colors[::2], s=0.5, marker='.')
        
        # Calculate bounding box corners
        cx, cy, cz = gt_box_center
        lx, ly, lz = gt_box_size
        
        x_min, x_max = cx - lx/2.0, cx + lx/2.0
        y_min, y_max = cy - ly/2.0, cy + ly/2.0
        z_min, z_max = cz - lz/2.0, cz + lz/2.0
        
        # Define the 8 corners of the bounding box
        corners = np.array([
            [x_min, y_min, z_min], [x_max, y_min, z_min],
            [x_max, y_max, z_min], [x_min, y_max, z_min],
            [x_min, y_min, z_max], [x_max, y_min, z_max],
            [x_max, y_max, z_max], [x_min, y_max, z_max]
        ])
        
        # Define the 12 edges connecting the corners
        edges = [
            [0, 1], [1, 2], [2, 3], [3, 0], # Bottom face
            [4, 5], [5, 6], [6, 7], [7, 4], # Top face
            [0, 4], [1, 5], [2, 6], [3, 7]  # Vertical pillars
        ]
        
        # Plot the bounding box edges
        for edge in edges:
            ax.plot(corners[edge, 0], corners[edge, 1], corners[edge, 2], 
                    color='red', linewidth=2.5)
            
        # Set axis limits to maintain a 1:1:1 aspect ratio (so the box doesn't distort)
        max_range = np.array([points[:,0].max()-points[:,0].min(), 
                              points[:,1].max()-points[:,1].min(), 
                              points[:,2].max()-points[:,2].min()]).max() / 2.0
        
        mid_x = (points[:,0].max() + points[:,0].min()) * 0.5
        mid_y = (points[:,1].max() + points[:,1].min()) * 0.5
        mid_z = (points[:,2].max() + points[:,2].min()) * 0.5
        
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        
        # Set labels and title
        ax.set_title(f"Target: '{text}'\nBox Center: {np.round(gt_box_center, 2)}")
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        plt.show()

    def visualize_point_cloud_interactive(self, idx):
        import plotly.graph_objects as go
        import numpy as np
        
        datapoint = self.__getitem__(idx)
        
        # Convert tensors back to numpy arrays
        point_cloud = datapoint["point_cloud"].numpy()
        gt_box_center = datapoint["gt_box_center"].numpy()
        gt_box_size = datapoint["gt_box_size"].numpy()
        text = datapoint["text"]
        
        # Extract coordinates and colors, filter padding
        points = point_cloud[:, :3]
        colors = point_cloud[:, 3:6]
        
        mask = np.any(points != 0, axis=1)
        points = points[mask]
        colors = colors[mask]
        
        # Subsample for even smoother rendering (optional)
        step = 2
        p_sub = points[::step]
        c_sub = colors[::step]
        
        # Plotly expects colors as an array of RGB strings
        color_strings = [f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})' 
                         for r, g, b in c_sub]
        
        # 1. Create the Point Cloud Trace
        trace_pc = go.Scatter3d(
            x=p_sub[:, 0], y=p_sub[:, 1], z=p_sub[:, 2],
            mode='markers',
            marker=dict(
                size=1.5,
                color=color_strings,
                opacity=0.8
            ),
            name='Point Cloud'
        )
        
        # 2. Calculate Bounding Box Corners
        cx, cy, cz = gt_box_center
        lx, ly, lz = gt_box_size
        
        x_min, x_max = cx - lx/2.0, cx + lx/2.0
        y_min, y_max = cy - ly/2.0, cy + ly/2.0
        z_min, z_max = cz - lz/2.0, cz + lz/2.0
        
        corners = np.array([
            [x_min, y_min, z_min], [x_max, y_min, z_min],
            [x_max, y_max, z_min], [x_min, y_max, z_min],
            [x_min, y_min, z_max], [x_max, y_min, z_max],
            [x_max, y_max, z_max], [x_min, y_max, z_max]
        ])
        
        # Plotly connects lines sequentially. We use None to break the line.
        edges = [
            [0, 1, 2, 3, 0], # Bottom face
            [4, 5, 6, 7, 4], # Top face
            [0, 4], [1, 5], [2, 6], [3, 7] # Pillars
        ]
        
        box_x, box_y, box_z = [], [], []
        for edge_seq in edges:
            for idx in edge_seq:
                box_x.append(corners[idx, 0])
                box_y.append(corners[idx, 1])
                box_z.append(corners[idx, 2])
            box_x.append(None) # Break line
            box_y.append(None)
            box_z.append(None)
            
        # 3. Create the Bounding Box Trace
        trace_box = go.Scatter3d(
            x=box_x, y=box_y, z=box_z,
            mode='lines',
            line=dict(color='red', width=4),
            name='Bounding Box'
        )
        
        # 4. Assemble and display the figure
        fig = go.Figure(data=[trace_pc, trace_box])
        
        fig.update_layout(
            title=f"Target: '{text}'<br>Box Center: {np.round(gt_box_center, 2)}",
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z',
                aspectmode='data' # This forces the 1:1:1 scale automatically!
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        
        fig.show()
        


In [3]:
train_scanrefer_dataset = ScanReferDataset(
    scanrefer_data_path="/home/avishka/sasika/grounding/3d/new_data/scanrefer/ScanRefer_filtered_train.json",
    scannet_dir="/home/avishka/sasika/grounding/3d/new_data/data/scannet",
    num_points=40000
)

val_scanrefer_dataset = ScanReferDataset(
    scanrefer_data_path="/home/avishka/sasika/grounding/3d/new_data/scanrefer/ScanRefer_filtered_val.json",
    scannet_dir="/home/avishka/sasika/grounding/3d/new_data/data/scannet",
    num_points=40000
)

In [4]:
train_scanrefer_dataset[0]

{'point_cloud': tensor([[6.0757, 8.2759, 1.3731, 0.5569, 0.5686, 0.5333],
         [1.5695, 6.8182, 1.4450, 0.5412, 0.4902, 0.3373],
         [1.8124, 4.9734, 0.0358, 0.8235, 0.7451, 0.6431],
         ...,
         [6.6771, 1.6628, 1.7196, 0.6118, 0.5373, 0.3961],
         [1.2453, 3.5889, 0.0503, 0.7490, 0.6824, 0.5843],
         [3.1063, 7.4879, 2.5968, 0.3843, 0.4314, 0.4706]]),
 'gt_box_center': tensor([2.1086, 0.7030, 1.0045]),
 'gt_box_size': tensor([0.8171, 1.2794, 1.9381]),
 'text': 'a white cabinet in the corner of the room . in the direction from the door and from the inside . it will be on the left , there is a small brown table on the left side of the cabinet and a smaller table on the right side of the cabinet'}

In [5]:
# train_scanrefer_dataset.visualize_point_cloud_interactive(0)

In [1]:
from transformers import AutoTokenizer, AutoModel

# The tokenizer is perfect
text_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Use AutoModel to get the raw hidden states (features)
# previously 
text_encoder = AutoModel.from_pretrained("distilbert-base-uncased")

/home/avishka/anaconda3/envs/sonata/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


- Point Encoder Output: You must ensure your point_encoder returns a sequence of features (e.g., $B \times N \times D$) rather than a single pooled vector per scene. If it returns a single vector, Cross-Attention won't be able to "look around" the scene, and you'll have to rely on simple concatenation, which performs poorly.

- 3D Augmentation: 3D bounding box regression overfits very quickly. You will eventually need to add data augmentations to your Dataset class (e.g., random point jittering, random dropping of points, random rotations along the Z-axis). If you rotate the point cloud, remember you must also rotate the gt_box_center.

## Model Architecture

In [6]:
ckpt_path: str = "POINTENCODER/weights/dVAE.pth"
device ="cuda"

In [7]:

from POINTENCODER.pointnet import PointnetTransformer


# 3. Initialize the Encoder
dvae_config = DictConfig({
    "encoder_dim": 256,
    "group_size": 32,
    "num_group": 64,
    "ckpt": ckpt_path,
    "freeze_encoder": True
})

transformer_config = DictConfig({
    "embed_dim": 768,
    "depth": 4,
    "num_heads": 12,
    "mlp_ratio": 4.0,
    "qkv_bias": False,
    "qk_scale": None,
    "drop_rate": 0.0,
    "attn_drop_rate": 0.0,
    "drop_path_rate": 0.1,
})

print("Initializing PointnetTransformer...")
point_encoder = PointnetTransformer(
    dvae_config=dvae_config,
    transformer_config=transformer_config
).to(device)
point_encoder.eval()

Initializing PointnetTransformer...
[Encoder] Loading pretrained encoder weights from POINTENCODER/weights/dVAE.pth
[Encoder] Successful Loading the ckpt for encoder from POINTENCODER/weights/dVAE.pth
[Encoder] Freezing encoder weights


PointnetTransformer(
  (encoder): Encoder(
    (first_conv): Sequential(
      (0): Conv1d(3, 128, kernel_size=(1,), stride=(1,))
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv1d(128, 256, kernel_size=(1,), stride=(1,))
    )
    (second_conv): Sequential(
      (0): Conv1d(512, 512, kernel_size=(1,), stride=(1,))
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv1d(512, 256, kernel_size=(1,), stride=(1,))
    )
  )
  (group_divider): Group()
  (reduce_dim): Linear(in_features=256, out_features=768, bias=True)
  (pos_embed): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=768, bias=True)
  )
  (blocks): TransformerEncoder(
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-0

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class VisualGroundingModel(nn.Module):
    def __init__(self, point_encoder, text_encoder, text_tokenizer, point_feat_dim=512, hidden_dim=256):
        super().__init__()
        
        # 1. 3D Spatial Encoder (Your PointnetTransformer)
        # We assume it outputs features of shape (Batch, Num_Points/Patches, point_feat_dim)
        self.point_encoder = point_encoder 
        
        # 2. Language Encoder (e.g., DistilBERT for speed)
        self.tokenizer = text_tokenizer
        self.text_encoder = text_encoder
        
        # Dimension Alignment Projections
        self.point_proj = nn.Linear(point_feat_dim, hidden_dim)
        self.text_proj = nn.Linear(self.text_encoder.config.hidden_size, hidden_dim)
        
        # 3. Cross-Modal Fusion
        # Points attend to text to find which spatial features match the words
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, 
            num_heads=8, 
            batch_first=True
        )
        
        # 4. Bounding Box Prediction Heads
        # Takes the fused global feature and predicts 3D coordinates
        self.center_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3) # Predicts cx, cy, cz
        )
        
        self.size_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3) # Predicts lx, ly, lz
        )

    def forward(self, point_cloud, raw_text):
        """
        point_cloud: Tensor of shape (B, N, 6)
        raw_text: List of strings
        """
        # --- Encode Text ---
        text_inputs = self.tokenizer(
            raw_text, padding=True, truncation=True, return_tensors="pt"
        ).to(point_cloud.device)
        
        t_outputs = self.text_encoder(**text_inputs)
        t_feats = t_outputs.last_hidden_state # (B, Seq_Len, Text_Dim)
        
        # --- Encode 3D Points ---
        # Assuming your encoder extracts patch/point features
        p_feats = self.point_encoder(point_cloud) # (B, Num_Patches, Point_Dim)
        
        # --- Align Dimensions ---
        p_feats = self.point_proj(p_feats)
        t_feats = self.text_proj(t_feats)
        
        # --- Cross-Modal Fusion ---
        # Query = 3D Points, Key/Value = Text Tokens
        fused_feats, attn_weights = self.cross_attention(
            query=p_feats, 
            key=t_feats, 
            value=t_feats
        ) # fused_feats shape: (B, Num_Patches, hidden_dim)
        
        # --- Global Pooling ---
        # Pool the sequence of point patches into a single feature vector per scene
        global_feat = torch.max(fused_feats, dim=1)[0] # Max pooling: (B, hidden_dim)
        
        # --- Predict ---
        pred_center = self.center_head(global_feat)
        pred_size = self.size_head(global_feat)
        
        return pred_center, pred_size

## Loss function

In [ ]:
import torch.nn.functional as F

def grounding_loss(pred_center, gt_center, pred_size, gt_size):
    # Smooth L1 Loss is robust to outliers in 3D space
    loss_center = F.smooth_l1_loss(pred_center, gt_center)
    loss_size = F.smooth_l1_loss(pred_size, gt_size)
    
    # You can weight these differently if one is dominating
    total_loss = loss_center + loss_size 
    return total_loss, loss_center, loss_size

## Training

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim


batch_size=8
epochs = 5
dataloader = DataLoader(train_scanrefer_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    # Initialize Model
vg_model = VisualGroundingModel(point_encoder=point_encoder, text_encoder=text_encoder, text_tokenizer=text_tokenizer).to(device)
    

def train_model(model, dataloader, epochs=10,device='cuda'):
    
    
    # Optimizer (AdamW is standard for Transformers)
    optimizer = optim.AdamW([
        {'params': model.point_encoder.parameters(), 'lr': 1e-4}, # Might want a lower LR if pre-trained
        {'params': model.text_encoder.parameters(), 'lr': 5e-5},
        {'params': model.point_proj.parameters()},
        {'params': model.text_proj.parameters()},
        {'params': model.cross_attention.parameters()},
        {'params': model.center_head.parameters()},
        {'params': model.size_head.parameters()}
    ], lr=1e-3)
    
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, batch in enumerate(dataloader):
            # Move data to device
            point_clouds = batch["point_cloud"].to(device)
            gt_centers = batch["gt_box_center"].to(device)
            gt_sizes = batch["gt_box_size"].to(device)
            texts = batch["text"] # List of strings, handled by tokenizer in forward pass
            

            # Inside your training loop:
            point_clouds = batch["point_cloud"].to(device) 

            # Slice to keep only XYZ (first 3 columns)
            xyz_only = point_clouds[:, :, :3] 

            # Now feed to the encoder
            p_feats = model(xyz_only)

            # Zero gradients
            optimizer.zero_grad()
            
            # Forward Pass
            pred_centers, pred_sizes = model(point_clouds, texts)
            
            # Compute Loss
            loss, l_center, l_size = grounding_loss(
                pred_centers, gt_centers, 
                pred_sizes, gt_sizes
            )
            
            # Backward Pass & Optimize
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch} | Batch {batch_idx}/{len(dataloader)} | "
                      f"Loss: {loss.item():.4f} (Center: {l_center.item():.4f}, Size: {l_size.item():.4f})")
                
        print(f"--- Epoch {epoch} Complete | Avg Loss: {epoch_loss/len(dataloader):.4f} ---")

    return model

In [ ]:
train_model(vg_model, dataloader, epochs=epochs, device='cuda')

In [ ]:
def compute_3d_iou(pred_center, pred_size, gt_center, gt_size):
    """
    Computes the 3D Intersection over Union (IoU) for Axis-Aligned Bounding Boxes.
    All inputs should be PyTorch Tensors of shape (Batch, 3).
    """
    # Convert Center/Size to Min/Max coordinates
    pred_min = pred_center - (pred_size / 2.0)
    pred_max = pred_center + (pred_size / 2.0)
    
    gt_min = gt_center - (gt_size / 2.0)
    gt_max = gt_center + (gt_size / 2.0)
    
    # Calculate Intersection limits
    inter_min = torch.max(pred_min, gt_min)
    inter_max = torch.min(pred_max, gt_max)
    
    # Calculate Intersection dimensions (clamp to 0 to avoid negative overlaps)
    inter_dims = torch.clamp(inter_max - inter_min, min=0.0)
    
    # Volume = Length * Width * Height
    inter_vol = inter_dims[:, 0] * inter_dims[:, 1] * inter_dims[:, 2]
    
    pred_vol = pred_size[:, 0] * pred_size[:, 1] * pred_size[:, 2]
    gt_vol = gt_size[:, 0] * gt_size[:, 1] * gt_size[:, 2]
    
    union_vol = pred_vol + gt_vol - inter_vol
    
    # Add a tiny epsilon to prevent division by zero
    iou = inter_vol / torch.clamp(union_vol, min=1e-6) 
    
    return iou # Returns a tensor of shape (Batch,)

In [ ]:
## Validation 

import os
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def train_and_validate(pointnet_encoder, train_dataset, val_dataset, epochs=10, batch_size=8, device='cuda', save_dir="checkpoints"):
    
    os.makedirs(save_dir, exist_ok=True)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    
    model = VisualGroundingModel(point_encoder=pointnet_encoder).to(device)
    
    optimizer = optim.AdamW([
        {'params': model.point_encoder.parameters(), 'lr': 1e-4}, 
        {'params': model.text_encoder.parameters(), 'lr': 5e-5},
        {'params': model.point_proj.parameters()},
        {'params': model.text_proj.parameters()},
        {'params': model.cross_attention.parameters()},
        {'params': model.center_head.parameters()},
        {'params': model.size_head.parameters()}
    ], lr=1e-3)
    
    best_val_iou = 0.0 # Track the best model
    
    for epoch in range(epochs):
        # ==========================================
        #               TRAINING PHASE
        # ==========================================
        model.train()
        train_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            # Only send XYZ to the encoder (drop RGB padding)
            point_clouds = batch["point_cloud"][:, :, :3].to(device) 
            gt_centers = batch["gt_box_center"].to(device)
            gt_sizes = batch["gt_box_size"].to(device)
            texts = batch["text"] 
            
            optimizer.zero_grad()
            
            pred_centers, pred_sizes = model(point_clouds, texts)
            
            loss, l_center, l_size = grounding_loss(
                pred_centers, gt_centers, pred_sizes, gt_sizes
            )
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            if batch_idx % 50 == 0:
                print(f"Epoch {epoch} [Train] | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
                
        avg_train_loss = train_loss / len(train_loader)
        
        # ==========================================
        #              VALIDATION PHASE
        # ==========================================
        model.eval() # Turn off dropout, batchnorm updates, etc.
        val_loss = 0.0
        val_iou_sum = 0.0
        iou_25_acc = 0 # How many boxes have IoU > 0.25
        total_samples = 0
        
        with torch.no_grad(): # Disable gradient tracking to save memory/speed
            for batch in val_loader:
                point_clouds = batch["point_cloud"][:, :, :3].to(device)
                gt_centers = batch["gt_box_center"].to(device)
                gt_sizes = batch["gt_box_size"].to(device)
                texts = batch["text"]
                
                pred_centers, pred_sizes = model(point_clouds, texts)
                
                # Validation Loss
                loss, _, _ = grounding_loss(pred_centers, gt_centers, pred_sizes, gt_sizes)
                val_loss += loss.item()
                
                # Validation Metrics (IoU)
                batch_ious = compute_3d_iou(pred_centers, pred_sizes, gt_centers, gt_sizes)
                val_iou_sum += batch_ious.sum().item()
                
                # Calculate accuracy at standard IoU threshold (0.25)
                iou_25_acc += (batch_ious >= 0.25).sum().item()
                total_samples += gt_centers.size(0)
                
        avg_val_loss = val_loss / len(val_loader)
        mean_iou = val_iou_sum / total_samples
        acc_25 = iou_25_acc / total_samples
        
        print(f"\n--- Epoch {epoch} Summary ---")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Val Mean IoU: {mean_iou:.4f} | Val Acc@0.25: {acc_25:.4f}")
        
        # ==========================================
        #               MODEL SAVING
        # ==========================================
        if mean_iou > best_val_iou:
            best_val_iou = mean_iou
            save_path = os.path.join(save_dir, "best_grounding_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_iou': best_val_iou,
            }, save_path)
            print(f"🌟 New best model saved! (IoU: {best_val_iou:.4f})")
        print("-" * 30 + "\n")

    return model